# Vantara Retail Solutions — Classical ML & Customer Segmentation
## Churn Classification, Forward CLV Regression & Persona Clustering

This notebook trains and evaluates:
1. **Six Classical Churn Classifiers**: Logistic Regression, Decision Tree, Random Forest, XGBoost, LightGBM, and KNN.
2. **CLV Regression Models**: Ridge, Random Forest, and LightGBM regressors predicting 90-day forward customer spend.
3. **Customer Segmentation**: K-Means and Gaussian Mixture Models (GMM) with Elbow/Silhouette optimization and persona profiling.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.models.data_split import prepare_data_splits

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
%matplotlib inline

### 1. Load Data & Prepare Stratified Splits

In [ ]:
df_features = pd.read_parquet("data/processed/customer_features.parquet")
splits_churn = prepare_data_splits(df_features, target_col="churn", test_size=0.15, val_size=0.15, random_seed=42)
print(f"Train cohort: {len(splits_churn.X_train)}, Val: {len(splits_churn.X_val)}, Test: {len(splits_churn.X_test)}")

### 2. Churn Classification Comparison

In [ ]:
with open("models_artifacts/churn_model_comparison.json", "r") as f:
    churn_comp = pd.DataFrame(json.load(f))

display(churn_comp[['Model', 'Test_ROC_AUC', 'Test_Recall', 'Test_Precision', 'Test_F1', 'Test_Accuracy']])

plt.figure(figsize=(10, 4))
sns.barplot(data=churn_comp, x='Test_ROC_AUC', y='Model', palette='Blues_r')
plt.axvline(0.80, color='red', linestyle='--', label='Target ROC-AUC >= 0.80')
plt.title('Held-Out Test ROC-AUC Across Classical Churn Classifiers')
plt.xlim(0.7, 0.85)
plt.legend()
plt.show()

### 3. CLV Regression Evaluation

In [ ]:
with open("models_artifacts/clv_model_comparison.json", "r") as f:
    clv_comp = pd.DataFrame(json.load(f))

display(clv_comp[['Model', 'Test_R2', 'Test_MAE', 'Test_RMSE']])

plt.figure(figsize=(8, 4))
sns.barplot(data=clv_comp, x='Test_R2', y='Model', palette='Greens_r')
plt.axvline(0.60, color='red', linestyle='--', label='Target R² >= 0.60')
plt.title('Held-Out Test R² for 90-Day Forward CLV')
plt.xlim(0.0, 1.0)
plt.legend()
plt.show()

### 4. Customer Segmentation & Persona Profiles

In [ ]:
with open("models_artifacts/segment_profiles.json", "r") as f:
    profiles_data = json.load(f)

seg_summary = pd.DataFrame.from_dict(profiles_data['segments'], orient='index')
display(seg_summary[['customer_count', 'share_of_cohort', 'mean_recency_days', 'mean_frequency_orders', 'mean_total_spend', 'churn_rate', 'forward_clv_90d']])